# MS-AR: Markov-Switching Autoregressive

Neste notebook, vamos aprender a estimar e interpretar um modelo **Markov-Switching AR** usando a biblioteca **archbox**.

O modelo MS-AR foi proposto por **Hamilton (1989)** para capturar mudancas de regime em series temporais.
A ideia central e que os parametros de um processo AR podem **mudar** de acordo com um estado latente
(nao observado) que segue uma **cadeia de Markov**.

**Aplicacao classica**: datacao de recessoes no PIB dos EUA.

**Conteudo:**
1. Cadeias de Markov ocultas
2. O modelo MS(2)-AR(1)
3. Probabilidades de regime suavizadas
4. Matriz de transicao
5. Datacao de recessoes
6. Escolha do numero de regimes

**Referencias:**
- Hamilton, J.D. (1989). *A New Approach to the Economic Analysis of Nonstationary Time Series and the Business Cycle*. Econometrica, 57(2), 357-384.
- Krolzig, H.-M. (1997). *Markov-Switching Vector Autoregressions*. Springer.
- Kim, C.-J. (1994). *Dynamic Linear Models with Markov-Switching*. Journal of Econometrics.

In [ ]:
import sys

sys.path.insert(0, '..')

import matplotlib.pyplot as plt
import pandas as pd

%matplotlib inline
plt.rcParams['figure.dpi'] = 100

# Carregar dados de crescimento do PIB dos EUA
data = pd.read_csv('../data/us_gdp_growth.csv', parse_dates=['date'], index_col='date')
gdp_growth = data['gdp_growth']

print(f"Periodo: {gdp_growth.index[0].date()} a {gdp_growth.index[-1].date()}")
print(f"Observacoes: {len(gdp_growth)} (trimestrais)")
print("\nEstatisticas descritivas:")
print(data.describe())

## 1. Cadeias de Markov ocultas

Uma **cadeia de Markov** e um processo estocastico onde o estado futuro depende apenas do estado atual
(propriedade de Markov). No contexto de modelos de regime, os estados sao **nao observados** (ocultos).

Seja $S_t \in \{1, 2, \ldots, K\}$ o estado (regime) no tempo $t$. A dinamica dos regimes e governada
pela **matriz de transicao** $\mathbf{P}$:

$$P(S_t = j \mid S_{t-1} = i) = p_{ij}$$

Para 2 regimes:

$$\mathbf{P} = \begin{pmatrix} p_{11} & p_{12} \\ p_{21} & p_{22} \end{pmatrix} = \begin{pmatrix} p_{11} & 1-p_{11} \\ 1-p_{22} & p_{22} \end{pmatrix}$$

onde $p_{11}$ e a probabilidade de **permanecer** no regime 1 e $p_{22}$ de permanecer no regime 2.

A **duracao esperada** de cada regime e:

$$E[D_i] = \frac{1}{1 - p_{ii}}$$

Por exemplo, se $p_{11} = 0.95$, o regime 1 dura em media $\frac{1}{1-0.95} = 20$ periodos.

In [ ]:
# TODO: Visualize o crescimento do PIB e identifique visualmente recessoes
# Dicas:
# - Plote a serie gdp_growth ao longo do tempo
# - Adicione uma linha horizontal em y=0 para separar expansao/recessao
# - Identifique visualmente periodos de crescimento negativo persistente
# - Use: fig, ax = plt.subplots(figsize=(14, 5))
#         ax.plot(gdp_growth.index, gdp_growth.values, color='black', lw=0.8)
#         ax.axhline(y=0, color='red', linestyle='--', alpha=0.5)
#         ax.set_title('Crescimento Trimestral do PIB dos EUA')
#         ax.set_ylabel('Crescimento (%)')
#         ax.grid(True, alpha=0.3)
#         plt.tight_layout()
#         plt.show()

## 2. O modelo MS(2)-AR(1)

O modelo **MS(2)-AR(1)** de Hamilton especifica:

$$y_t = \mu_{S_t} + \phi_{S_t} (y_{t-1} - \mu_{S_{t-1}}) + \epsilon_t, \quad \epsilon_t \sim N(0, \sigma^2_{S_t})$$

onde $S_t \in \{1, 2\}$ indica o regime ativo no tempo $t$.

Cada regime tem seus proprios parametros:
- **Regime 1 (expansao)**: $\mu_1 > 0$ (crescimento positivo), $\sigma_1$ baixo
- **Regime 2 (recessao)**: $\mu_2 < 0$ (crescimento negativo), $\sigma_2$ alto

### Estimacao: Hamilton Filter + EM

A estimacao segue o **algoritmo EM** (Expectation-Maximization):

1. **E-step (Hamilton Filter)**: calcula $P(S_t = j \mid \mathcal{Y}_t)$ recursivamente
2. **Smoothing (Kim Smoother)**: calcula $P(S_t = j \mid \mathcal{Y}_T)$ usando toda a amostra
3. **M-step**: atualiza os parametros usando as probabilidades suavizadas como pesos

In [ ]:
# TODO: Estime MS(2)-AR(1) com archbox
# Dicas:
# - Crie o modelo:
#     model = MarkovSwitchingAR(
#         endog=gdp_growth.values,
#         k_regimes=2,
#         order=1,
#         switching_mean=True,
#         switching_variance=True,
#     )
# - Ajuste: results = model.fit(method='em', maxiter=500, verbose=True)
# - Exiba o resumo: print(results.summary())
# - Verifique convergencia: print(f'Convergiu: {results.converged}')
# - Verifique log-likelihood: print(f'Log-likelihood: {results.loglike:.4f}')

## 3. Probabilidades de regime suavizadas (smoothed probabilities)

As **probabilidades suavizadas** sao a inferencia mais completa sobre os regimes, pois
utilizam **toda a informacao da amostra**:

$$P(S_t = j \mid \mathcal{Y}_T) \quad \text{para } t = 1, \ldots, T$$

Existem 3 tipos de probabilidades:

| Tipo | Formula | Usa informacao ate |
|------|---------|--------------------|
| **Predita** | $P(S_t = j \mid \mathcal{Y}_{t-1})$ | $t-1$ |
| **Filtrada** | $P(S_t = j \mid \mathcal{Y}_t)$ | $t$ |
| **Suavizada** | $P(S_t = j \mid \mathcal{Y}_T)$ | $T$ (toda amostra) |

As probabilidades suavizadas sao calculadas pelo **Kim smoother** (Kim, 1994),
que aplica um passo para tras (backward) apos o Hamilton filter (forward).

In [ ]:
# TODO: Plote probabilidades de regime suavizadas
# Dicas:
# - Probabilidades suavizadas: results.smoothed_probs  (shape: T x k_regimes)
# - Use a funcao auxiliar:
#     fig = plot_regime_probabilities(
#         dates=gdp_growth.index,
#         series=gdp_growth.values,
#         probabilities=results.smoothed_probs,
#         regime_labels=['Expansao', 'Recessao'],
#         title='MS(2)-AR(1): PIB dos EUA com Probabilidades de Regime',
#     )
#     plt.show()
# - Observe como as probabilidades de recessao aumentam em periodos de queda do PIB

## 4. Matriz de transicao

A **matriz de transicao** $\mathbf{P}$ resume a dinamica entre regimes:

$$\mathbf{P} = \begin{pmatrix} p_{11} & p_{12} \\ p_{21} & p_{22} \end{pmatrix}$$

onde $p_{ij} = P(S_t = j \mid S_{t-1} = i)$.

**Interpretacao economica:**
- $p_{11}$: probabilidade de continuar em expansao $\Rightarrow$ duracao esperada = $\frac{1}{1-p_{11}}$
- $p_{22}$: probabilidade de continuar em recessao $\Rightarrow$ duracao esperada = $\frac{1}{1-p_{22}}$

As **probabilidades ergoticas** (longo prazo) sao:

$$\pi_1 = \frac{1 - p_{22}}{2 - p_{11} - p_{22}}, \quad \pi_2 = 1 - \pi_1$$

Estas indicam a fracao do tempo que a economia passa em cada regime.

In [ ]:
# TODO: Extraia e interprete a matriz de transicao
# Dicas:
# - Matriz de transicao: P = results.transition_matrix
# - Plote como heatmap:
#     fig = plot_transition_matrix(
#         P,
#         regime_labels=['Expansao', 'Recessao'],
#         title='Matriz de Transicao Estimada',
#     )
#     plt.show()
#
# - Calcule duracoes esperadas:
#     duracoes = results.expected_durations()
#     print(f'Duracao esperada da expansao: {duracoes[0]:.1f} trimestres')
#     print(f'Duracao esperada da recessao: {duracoes[1]:.1f} trimestres')
#
# - Calcule probabilidades ergoticas:
#     ergodic = results.ergodic_probabilities()
#     print(f'Fracao do tempo em expansao: {ergodic[0]:.1%}')
#     print(f'Fracao do tempo em recessao: {ergodic[1]:.1%}')

## 5. Datacao de recessoes

Uma aplicacao classica do MS-AR e a **datacao automatica de recessoes**.
Podemos classificar cada observacao no regime mais provavel:

$$\hat{S}_t = \arg\max_j P(S_t = j \mid \mathcal{Y}_T)$$

Hamilton (1989) mostrou que o modelo MS(2)-AR(4) aplicado ao PIB dos EUA
reproduz com notavel precisao as datacoes de recessao do **NBER**
(National Bureau of Economic Research), sem usar informacao ex-post.

Isso sugere que as recessoes sao um fenomeno real (mudanca de regime),
nao apenas variacoes aleatorias no crescimento.

In [ ]:
# TODO: Classifique cada observacao no regime mais provavel
# Dicas:
# - Classificacao: regimes = results.classify(threshold=0.5)
# - Plote com medias por regime:
#     regime_params = results.regime_params
#     mu = {r: params['mu'] for r, params in regime_params.items()}
#     fig = plot_regime_means(
#         dates=gdp_growth.index,
#         series=gdp_growth.values,
#         regimes=regimes,
#         mu=mu,
#         regime_labels=['Expansao', 'Recessao'],
#         title='Datacao de Regimes: Expansao vs Recessao',
#     )
#     plt.show()
#
# - Conte o numero de trimestres em recessao:
#     n_recessao = np.sum(regimes == 2)  # ou == 1 dependendo da rotulagem
#     print(f'Trimestres em recessao: {n_recessao} ({n_recessao/len(regimes):.1%})')
#
# - Compare com os regimes verdadeiros do dataset sintetico:
#     if 'true_regime' in data.columns:
#         accuracy = np.mean(regimes == data['true_regime'].values)
#         print(f'Acuracia vs regimes verdadeiros: {accuracy:.1%}')

## 6. Escolha do numero de regimes

Na pratica, precisamos decidir quantos regimes usar. Opcoes comuns:

- **MS(2)**: expansao vs recessao (mais comum)
- **MS(3)**: expansao, recessao moderada, recessao severa

**Criterios de selecao:**
- **AIC** (Akaike Information Criterion): $AIC = -2\ell + 2k$
- **BIC** (Bayesian Information Criterion): $BIC = -2\ell + k\ln(T)$

onde $\ell$ e a log-verossimilhanca e $k$ o numero de parametros.

**Nota importante**: O teste de razao de verossimilhanca (LRT) **nao** tem distribuicao
$\chi^2$ padrao neste contexto, pois sob $H_0$ alguns parametros nao sao identificados
(problema de Davies, 1987). Por isso, usamos criterios de informacao.

In [ ]:
# TODO: Estime MS(3)-AR(1) e compare com MS(2) via AIC
# Dicas:
# - Estime MS(3)-AR(1):
#     model_3 = MarkovSwitchingAR(
#         endog=gdp_growth.values,
#         k_regimes=3,
#         order=1,
#         switching_mean=True,
#         switching_variance=True,
#     )
#     results_3 = model_3.fit(method='em', maxiter=500, verbose=True)
#
# - Compare criterios de informacao:
#     print('Comparacao MS(2) vs MS(3):')
#     print(f'  MS(2): AIC={results.aic:.2f}, BIC={results.bic:.2f}')
#     print(f'  MS(3): AIC={results_3.aic:.2f}, BIC={results_3.bic:.2f}')
#
# - Plote probabilidades do MS(3):
#     fig = plot_regime_probabilities(
#         dates=gdp_growth.index,
#         series=gdp_growth.values,
#         probabilities=results_3.smoothed_probs,
#         regime_labels=['Regime 1', 'Regime 2', 'Regime 3'],
#         title='MS(3)-AR(1): Tres Regimes',
#     )
#     plt.show()
#
# - Interprete: o terceiro regime captura algo novo ou e redundante?

## Conclusao

Neste notebook, aprendemos:

- O conceito de **cadeias de Markov ocultas** e sua aplicacao em econometria
- Como estimar um modelo **MS(2)-AR(1)** com o algoritmo EM (Hamilton filter + Kim smoother)
- Como interpretar as **probabilidades de regime suavizadas** $P(S_t = j \mid \mathcal{Y}_T)$
- Como extrair e interpretar a **matriz de transicao** (duracoes esperadas, probabilidades ergoticas)
- Como usar o modelo para **datacao automatica de recessoes**
- Como escolher o **numero de regimes** usando AIC/BIC

No proximo notebook, estendemos essa abordagem para sistemas **multivariados** com o **MS-VAR**.

### Referencias

- Hamilton, J.D. (1989). A New Approach to the Economic Analysis of Nonstationary Time Series and the Business Cycle. *Econometrica*, 57(2), 357-384.
- Kim, C.-J. (1994). Dynamic Linear Models with Markov-Switching. *Journal of Econometrics*, 60, 1-22.
- Krolzig, H.-M. (1997). *Markov-Switching Vector Autoregressions*. Springer.
- Haas, M., Mittnik, S., & Paolella, M.S. (2004). A New Approach to Markov-Switching GARCH Models. *Journal of Financial Econometrics*, 2(4), 493-530.